# RESTOPS · Make metrics reliable before explaining performance

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src.common import read, TABLEAU, REPORTS
from src.reporting import style
style()
pd.set_option("display.max_columns", 12)


**Business question:** Can Finance trust the sales and cost populations?

Review the cleaning audit and complete-order quarantine. Repairs require visible business rules; missing answers and unidentified customers retain their meaning.

In [2]:
audit = pd.read_csv(REPORTS / "exports/data_quality_audit.csv")
audit[audit.affected_rows > 0]

,table,rule,affected_rows,action
2,transactions,Exact duplicate records,2300,Removed
3,labour,Exact duplicate records,180,Removed
4,customer_feedback,Exact duplicate records,100,Removed
13,products,Category aliases,2,Mapped to canonical categories
14,products,Category inconsistent with approved menu taxonomy,2,Corrected using checked-in business reference
15,transactions,Store name inconsistent with restaurant_id,2600,Replaced from restaurant master
17,transactions,Invalid product foreign key,800,Quarantine complete affected order
18,transactions,Invalid timestamp,650,Quarantine complete affected order
20,transactions,Invalid quantity,350,Quarantine complete affected order
21,transactions,Invalid monetary values,250,Quarantine complete affected order


In [3]:
quality = json.loads((REPORTS / "quality_summary.json").read_text())
pd.Series({k:v for k,v in quality.items() if k != "clean_rows"})

raw_lines_after_dedup     2111602.00
clean_lines               2105626.00
clean_orders               794925.00
quarantined_orders           2049.00
clean_revenue            36444593.80
quarantined_revenue       -125469.65
dtype: float64

### Quarantine full baskets
Removing only damaged product lines would retain incomplete orders and distort transaction value and product mix. Companion lines therefore leave the analytical sales population too.

In [4]:
pd.read_csv(ROOT / "data/processed/quarantine_transactions.csv", nrows=6)[["order_id", "product_id", "quantity", "quality_reason"]]

,order_id,product_id,quantity,quality_reason
0,112,7.0,1,Companion line in rejected order
1,112,1.0,1,Companion line in rejected order
2,112,16.0,1,Companion line in rejected order
3,112,NaN,1,Invalid product foreign key;
4,425,16.0,1,Companion line in rejected order
5,425,NaN,1,Invalid product foreign key;


In [5]:
labour = read("labour")
labour.groupby("rate_imputed").agg(shifts=("shift_id", "size"), paid_hours=("paid_hours", "sum"))

,shifts,paid_hours
rate_imputed,,
0,90428,374587.0
1,130,536.0


In [6]:
products = read("products")
products.groupby("category").product_id.count()

category
Desserts    4
Drinks      5
Mains       8
Salads      3
Sides       4
Name: product_id, dtype: int64

**Decision:** Investigate source exceptions before refreshing management reports. Category repairs use the checked-in approved menu taxonomy, not a hidden clean dataset. Retained survey nulls are excluded from score denominators.